In [1]:
import os
from pyspark.sql import SparkSession

# Khởi tạo Spark Session cho Lab 4
spark = (SparkSession.builder
    .appName("Lab4_SparkML_Home")
    # Nạp gói Kafka (bắt buộc để làm Ex 0)
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1")
    # Kích hoạt AQE theo đặc tả trang 2 của Lab 4
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .master("local[*]") 
    .getOrCreate())

print("--- Spark Session cho Lab 4 đã sẵn sàng! ---")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/14 07:49:11 WARN Utils: Your hostname, HungThinh, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/14 07:49:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/thinh/BigData/bigdata_lab/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/thinh/.ivy2.5.2/cache
The jars for the packages stored in: /home/thinh/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5efdc8a3-074e-4fd9-a356-b34ff0519568;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	foun

--- Spark Session cho Lab 4 đã sẵn sàng! ---


In [3]:
from confluent_kafka.admin import AdminClient, NewTopic

admin_client = AdminClient({'bootstrap.servers': 'localhost:9092,localhost:9192,localhost:9292'})
topic_names = ['Lab1_movies', 'Lab1_ratings', 'Lab1_tags']
admin_client.delete_topics(topic_names, operation_timeout=10)
new_topics = [NewTopic(topic=name, num_partitions=1, replication_factor=1) for name in topic_names]
fs = admin_client.create_topics(new_topics)

for topic, f in fs.items():
    try:
        f.result()
        print(f"Topic '{topic}' đã được khởi tạo thành công!")
    except Exception as e:
        print(f"Bỏ qua/Lỗi tại {topic}: {e}")

Topic 'Lab1_movies' đã được khởi tạo thành công!
Topic 'Lab1_ratings' đã được khởi tạo thành công!
Topic 'Lab1_tags' đã được khởi tạo thành công!


In [4]:
import kagglehub
from pyspark.sql.functions import to_json, struct

# 1. Tải dataset từ Kaggle
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

# 2. Đọc file CSV bằng Spark 
df_r = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_m = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_t = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

# 3. Hàm đẩy dữ liệu vào Kafka
def push_to_kafka(df, topic):
    df.selectExpr("to_json(struct(*)) AS value") \
      .write.format("kafka") \
      .option("kafka.bootstrap.servers", "localhost:9092,localhost:9192,localhost:9292") \
      .option("topic", topic) \
      .save()
    print(f"Đã nạp dữ liệu cho {topic}")

# Thực hiện nạp cho 3 topic 
push_to_kafka(df_m, "Lab1_movies")
push_to_kafka(df_r, "Lab1_ratings")
push_to_kafka(df_t, "Lab1_tags")

/home/thinh/BigData/bigdata_lab/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đã nạp dữ liệu cho Lab1_movies


Đã nạp dữ liệu cho Lab1_ratings
Đã nạp dữ liệu cho Lab1_tags


In [12]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col

# 1. Định nghĩa Schema cho movies, ratings và tags theo đặc tả Lab 4
movie_schema = StructType([
    StructField("movieId", IntegerType()), 
    StructField("title", StringType()), 
    StructField("genres", StringType())
])
rating_schema = StructType([
    StructField("userId", IntegerType()), 
    StructField("movieId", IntegerType()), 
    StructField("rating", DoubleType()), 
    StructField("timestamp", LongType())
])
tag_schema = StructType([
    StructField("userId", IntegerType()), 
    StructField("movieId", IntegerType()), 
    StructField("tag", StringType()), 
    StructField("timestamp", LongType())
])

# 2. Hàm load dữ liệu từ Kafka vào DataFrame
def load_kafka(topic, schema):
    return (spark.read.format("kafka")
        .option("kafka.bootstrap.servers", "localhost:9092,localhost:9192,localhost:9292")
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .load()
        .select(from_json(col("value").cast("string"), schema).alias("data"))
        .select("data.*"))

# 3. Chuyển đổi dữ liệu sang định dạng chuẩn
df_movies = load_kafka("Lab1_movies", movie_schema)
df_ratings = load_kafka("Lab1_ratings", rating_schema)
df_tags = load_kafka("Lab1_tags", tag_schema)

print("--- Exercise 0: Hoàn thành! ---")
df_movies.show(5)
df_ratings.show(5)
df_tags.show(5)

--- Exercise 0: Hoàn thành! ---
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows
+------+-------+---------------+----------+
|userId|movieId|            tag| timestamp|
+------+-------+---------------+----------+
|     2|  6075

Exercise1

In [14]:
from pyspark.sql.functions import avg, count, when, col, concat_ws, collect_list

# 1. Tạo nhãn (Labeling): 1 nếu rating >= 4, ngược lại là 0
df_labeled = df_ratings.withColumn("label", when(col("rating") >= 4, 1).otherwise(0))

# 2. Tính toán đặc trưng số (Numerics): Thống kê User và Movie
movie_stats = df_ratings.groupBy("movieId").agg(
    avg("rating").alias("movie_avg_rating"),
    count("rating").alias("movie_rating_count")
)

user_stats = df_ratings.groupBy("userId").agg(
    avg("rating").alias("user_avg_rating"),
    count("rating").alias("user_rating_count")
)

# 3. Xử lý văn bản (Text): Gom nhóm tags theo User-Movie và kết hợp với Title
user_movie_tags = df_tags.groupBy("userId", "movieId").agg(
    concat_ws(" ", collect_list("tag")).alias("user_tags")
)

# 4. Join tất cả lại thành một DataFrame huấn luyện duy nhất
train_df = df_labeled.join(df_movies, "movieId") \
    .join(movie_stats, "movieId") \
    .join(user_stats, "userId") \
    .join(user_movie_tags, ["userId", "movieId"], "left") \
    .fillna("") # Thay thế null bằng chuỗi rỗng cho phần tags

# Kết hợp Title và Tags thành một cột văn bản duy nhất
train_df = train_df.withColumn("text_metadata", concat_ws(" ", col("title"), col("user_tags")))

print("Dữ liệu chuẩn bị cho huấn luyện:")
train_df.select("userId", "movieId", "text_metadata", "genres", "label").show(5)

Dữ liệu chuẩn bị cho huấn luyện:


+------+-------+--------------------+--------------------+-----+
|userId|movieId|       text_metadata|              genres|label|
+------+-------+--------------------+--------------------+-----+
|     1|   1580|Men in Black (a.k...|Action|Comedy|Sci-Fi|    0|
|     1|   2366|   King Kong (1933) |Action|Adventure|...|    1|
|     4|   1580|Men in Black (a.k...|Action|Comedy|Sci-Fi|    0|
|     4|   3175|Galaxy Quest (1999) |Adventure|Comedy|...|    0|
|    10|   1088|Dirty Dancing (19...|Drama|Musical|Rom...|    0|
+------+-------+--------------------+--------------------+-----+
only showing top 5 rows


In [16]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, HashingTF, StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression

# Bước 1: Xử lý Text (Title + Tags)
tokenizer = Tokenizer(inputCol="text_metadata", outputCol="words")
hashingTF = HashingTF(inputCol="words", outputCol="text_features", numFeatures=100)

# Bước 2: Xử lý Categorical (Genres) - Dùng StringIndexer cho đơn giản
genre_indexer = StringIndexer(inputCol="genres", outputCol="genre_idx", handleInvalid="keep")

# Bước 3: Gom nhóm tất cả đặc trưng vào một Vector
assembler = VectorAssembler(
    inputCols=["text_features", "genre_idx", "movie_avg_rating", "user_avg_rating"],
    outputCol="raw_features"
)

# Bước 4: Chuẩn hóa dữ liệu (Scaling)
scaler = StandardScaler(inputCol="raw_features", outputCol="features")

# Bước 5: Khai báo mô hình Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

# Tạo Pipeline hoàn chỉnh
pipeline = Pipeline(stages=[tokenizer, hashingTF, genre_indexer, assembler, scaler, lr])

# Chia dữ liệu Train/Test
train_data, test_data = train_df.randomSplit([0.8, 0.2], seed=42)

# Huấn luyện mô hình
model = pipeline.fit(train_data)
print("--- Đã huấn luyện xong mô hình! ---")

26/05/14 08:07:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/14 08:07:41 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


--- Đã huấn luyện xong mô hình! ---


In [17]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Dự đoán trên tập Test
predictions = model.transform(test_data)

# 1. Tính AUC
evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc = evaluator_auc.evaluate(predictions)

# 2. Tính F1-Score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
f1 = evaluator_f1.evaluate(predictions)

print(f"--- Kết quả đánh giá ---")
print(f"AUC (Primary): {auc:.4f}")
print(f"F1 Score: {f1:.4f}")

# 3. Tạo Ma trận nhầm lẫn (Confusion Matrix) 2x2
print("\nMa trận nhầm lẫn (Confusion Matrix):")
predictions.groupBy("label", "prediction").count().show()

--- Kết quả đánh giá ---
AUC (Primary): 0.8158
F1 Score: 0.7339

Ma trận nhầm lẫn (Confusion Matrix):


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0| 2579|
|    0|       0.0| 7706|
|    1|       1.0| 7160|
|    0|       1.0| 2813|
+-----+----------+-----+



In [19]:
# Lấy trọng số từ mô hình Logistic Regression
lr_model = model.stages[-1]
weights = lr_model.coefficients.toArray()

# Lưu ý: Do dùng HashingTF nên việc ánh xạ lại từng từ khá phức tạp. 
# Ở mức độ Lab, ta có thể liệt kê các đặc trưng có trọng số cao nhất.
import numpy as np

top_positive = np.argsort(weights)[-10:]
top_negative = np.argsort(weights)[:10]

print("Top 10 Positive Feature Indices:", top_positive)
print("Top 10 Negative Feature Indices:", top_negative)

# --- INSIGHTS CHO EXERCISE 1 ---
print("\nInsights for Exercise 1:")
print("1. AUC đạt mức ổn định cho thấy sự kết hợp giữa văn bản (tags) và thống kê (avg rating) mang lại độ chính xác cao.")
print("2. Ma trận nhầm lẫn cho thấy mô hình có xu hướng dự đoán đúng các phim được đánh giá cao tốt hơn các phim điểm thấp.")
print("3. Các chỉ số về avg_user_rating thường là tín hiệu quan trọng nhất vì nó phản ánh gu cá nhân của người dùng.")

Top 10 Positive Feature Indices: [ 90  48  78  82  35  60  36  81 102 101]
Top 10 Negative Feature Indices: [ 7 71 24 59 39 57  6 95 67 20]

Insights for Exercise 1:
1. AUC đạt mức ổn định cho thấy sự kết hợp giữa văn bản (tags) và thống kê (avg rating) mang lại độ chính xác cao.
2. Ma trận nhầm lẫn cho thấy mô hình có xu hướng dự đoán đúng các phim được đánh giá cao tốt hơn các phim điểm thấp.
3. Các chỉ số về avg_user_rating thường là tín hiệu quan trọng nhất vì nó phản ánh gu cá nhân của người dùng.


Exercise2